# week 1

# 🎙️ Speech Emotion Recognition Using Machine Learning
---
This notebook builds a complete Speech Emotion Recognition (SER) system step by step.

The system listens to audio recordings and detects the emotion of the speaker —
such as happy, sad, angry, neutral, fearful, and more.

We follow a clear pipeline:
1. Load and explore the datasets
2. Clean and prepare the audio files
3. Extract features from each audio file
4. Train and compare four models
5. Evaluate and present the final results

All four datasets are free and available on Kaggle.
All experiments run on Kaggle free GPU — no local setup needed.

## Section 1 — Import Libraries
---
In this section we import all the Python libraries we need for the entire project.

- **librosa** — reads audio files and extracts features like MFCC
- **numpy and pandas** — handle data and organise it into tables
- **scikit-learn** — used for the SVM baseline model and data scaling
- **PyTorch** — used to build CNN, CNN-LSTM and Attention deep learning models
- **matplotlib and seaborn** — used to plot graphs and confusion matrices
- **os and glob** — used to navigate folders and find audio files

In [ ]:
# ─────────────────────────────────────────────
# SECTION 1 — IMPORT LIBRARIES
# ─────────────────────────────────────────────

# Audio processing
import librosa
import librosa.display

# Data handling
import numpy as np
import pandas as pd

# File and folder navigation
import os
import glob

# Visualisation
import matplotlib.pyplot as plt
import seaborn as sns

# Machine learning utilities
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import (accuracy_score, f1_score,
                             classification_report,
                             confusion_matrix)

# Deep learning
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

# Progress bar
from tqdm import tqdm

# Suppress warnings for clean output
import warnings
warnings.filterwarnings('ignore')

# Check GPU availability
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")
print("All libraries imported successfully.")

In [ ]:
# ─────────────────────────────────────────────────────────────
# MASTER OUTPUT FOLDER SETUP
# Run this once at the start — creates all section folders
# ─────────────────────────────────────────────────────────────

import os

# Define all section output folders
folders = [
    '/kaggle/working/outputs',
    '/kaggle/working/outputs/section2_datasets',
    '/kaggle/working/outputs/section3_features',
    '/kaggle/working/outputs/section4_svm',
    '/kaggle/working/outputs/section5_cnn',
    '/kaggle/working/outputs/section6_cnn_lstm',
    '/kaggle/working/outputs/section7_attention',
    '/kaggle/working/outputs/section8_results',
]

for folder in folders:
    os.makedirs(folder, exist_ok=True)

print("All output folders created successfully.")
print()
for folder in folders:
    print(f"  ✓  {folder}")

## Section 2 — Load Datasets
---
We use four publicly available speech emotion datasets. Each dataset contains
audio recordings of actors speaking with different emotions.

| Dataset | Speakers | Emotions | Files |
|---------|----------|----------|-------|
| RAVDESS | 24 professional actors | 8 emotions | ~1,440 |
| CREMA-D | 91 actors | 6 emotions | 7,442 |
| TESS | 2 female speakers | 7 emotions | 2,800 |
| SAVEE | 4 male speakers | 7 emotions | 480 |

All datasets are already added to this notebook via Kaggle.
We load all audio file paths and their emotion labels into one combined dataframe.

The emotion labels across all datasets are mapped to these common categories:
angry, disgust, fear, happy, neutral, sad, surprised, calm.

In [ ]:
# ── Label extractor functions ──

import os
import glob
import pandas as pd

def get_ravdess_label(filepath):
    filename = os.path.basename(filepath)
    parts = filename.split('-')
    if len(parts) < 3:
        return None
    try:
        emotion_code = int(parts[2])
    except ValueError:
        return None
    emotion_map = {
        1: 'neutral', 2: 'calm',     3: 'happy', 4: 'sad',
        5: 'angry',   6: 'fearful',  7: 'disgust', 8: 'surprised'
    }
    return emotion_map.get(emotion_code, None)

def get_cremad_label(filepath):
    filename = os.path.basename(filepath)
    parts = filename.split('_')
    if len(parts) < 3:
        return None
    emotion_code = parts[2]
    emotion_map = {
        'ANG': 'angry',   'DIS': 'disgust', 'FEA': 'fearful',
        'HAP': 'happy',   'NEU': 'neutral', 'SAD': 'sad'
    }
    return emotion_map.get(emotion_code, None)

def get_tess_label(filepath):
    folder = os.path.basename(os.path.dirname(filepath)).lower()
    emotion_map = {
        'angry':   'angry',   'disgust': 'disgust',
        'fear':    'fearful', 'happy':   'happy',
        'neutral': 'neutral', 'sad':     'sad',
        'ps':      'surprised'
    }
    for key in emotion_map:
        if key in folder:
            return emotion_map[key]
    return None

def get_savee_label(filepath):
    filename = os.path.basename(filepath).lower()
    parts = filename.split('_')
    if len(parts) < 2:
        return None
    code = parts[1][:2].strip()
    emotion_map = {
        'a':  'angry',   'd':  'disgust', 'f':  'fearful',
        'h':  'happy',   'n':  'neutral', 'sa': 'sad',
        'su': 'surprised'
    }
    return emotion_map.get(code, emotion_map.get(code[0], None))

print("Label functions defined successfully.")

In [ ]:
# ─────────────────────────────────────────────
# SECTION 2 — LOAD DATASETS (FIXED PATHS)
# ─────────────────────────────────────────────

def load_all_datasets():
    data = []

    # ── RAVDESS ──
    ravdess_files = glob.glob(
        '/kaggle/input/datasets/uwrfkaggler/ravdess-emotional-speech-audio/**/*.wav',
        recursive=True
    )
    for f in ravdess_files:
        label = get_ravdess_label(f)
        if label:
            data.append({'path': f, 'emotion': label, 'source': 'RAVDESS'})

    # ── CREMA-D ──
    # CREMA-D is inside the dmitrybabko combined pack
    cremad_files = glob.glob(
        '/kaggle/input/datasets/dmitrybabko/speech-emotion-recognition-en/Crema/**/*.wav',
        recursive=True
    )
    for f in cremad_files:
        label = get_cremad_label(f)
        if label:
            data.append({'path': f, 'emotion': label, 'source': 'CREMAD'})

    # ── TESS ──
    tess_files = glob.glob(
        '/kaggle/input/datasets/ejlok1/toronto-emotional-speech-set-tess/**/*.wav',
        recursive=True
    )
    for f in tess_files:
        label = get_tess_label(f)
        if label:
            data.append({'path': f, 'emotion': label, 'source': 'TESS'})

    # ── SAVEE ──
    savee_files = glob.glob(
        '/kaggle/input/datasets/dmitrybabko/speech-emotion-recognition-en/Savee/**/*.wav',
        recursive=True
    )
    for f in savee_files:
        label = get_savee_label(f)
        if label:
            data.append({'path': f, 'emotion': label, 'source': 'SAVEE'})

    return pd.DataFrame(data)


# Load everything
df = load_all_datasets()

print(f"Total audio files loaded: {len(df)}")
print(f"\nFiles per dataset:")
print(df['source'].value_counts())
print(f"\nEmotion distribution:")
print(df['emotion'].value_counts())

In [ ]:
# ── 2.3 Visualise emotion distribution ──

plt.figure(figsize=(12, 4))

# Emotion count
plt.subplot(1, 2, 1)
df['emotion'].value_counts().plot(kind='bar', color='#2e7d32')
plt.title('Emotion Distribution (All Datasets)')
plt.xlabel('Emotion')
plt.ylabel('Count')
plt.xticks(rotation=45)

# Dataset count
plt.subplot(1, 2, 2)
df['source'].value_counts().plot(kind='bar', color='#1b5e20')
plt.title('Files per Dataset')
plt.xlabel('Dataset')
plt.ylabel('Count')
plt.xticks(rotation=45)

plt.tight_layout()
plt.show()

In [ ]:
# ── Save Section 2 Outputs ──

import matplotlib.pyplot as plt
import seaborn as sns

# Plot 1 — Emotion distribution
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

df['emotion'].value_counts().plot(
    kind='bar', color='#2e7d32', ax=axes[0]
)
axes[0].set_title('Emotion Distribution (All Datasets)')
axes[0].set_xlabel('Emotion')
axes[0].set_ylabel('Count')
axes[0].tick_params(axis='x', rotation=45)

df['source'].value_counts().plot(
    kind='bar', color='#1b5e20', ax=axes[1]
)
axes[1].set_title('Files Per Dataset')
axes[1].set_xlabel('Dataset')
axes[1].set_ylabel('Count')
axes[1].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.savefig(
    '/kaggle/working/outputs/section2_datasets/dataset_distribution.png',
    dpi=150
)
plt.close()

# Save dataset summary CSV
df.groupby(['source', 'emotion']).size()\
  .reset_index(name='count')\
  .to_csv(
      '/kaggle/working/outputs/section2_datasets/dataset_summary.csv',
      index=False
  )

# Save JSON summary
import json
summary_s2 = {
    'total_files'        : int(len(df)),
    'files_per_dataset'  : df['source'].value_counts().to_dict(),
    'emotion_distribution': df['emotion'].value_counts().to_dict()
}
with open('/kaggle/working/outputs/section2_datasets/dataset_info.json', 'w') as f:
    json.dump(summary_s2, f, indent=4)

print("Section 2 outputs saved.")
print("  dataset_distribution.png")
print("  dataset_summary.csv")
print("  dataset_info.json")

# week 2

## Section 3 — Data Preprocessing
---
Raw audio files cannot be used directly for training. We need to clean and
prepare them first. This section does three things:

### 3.1 — Silence Trimming
Each audio file may have silence at the beginning or end.
We remove this silence so the model only learns from actual speech.

### 3.2 — Amplitude Normalisation
Different recordings have different volume levels.
We normalise all files to the same volume so the model is not confused
by loud or quiet recordings.

### 3.3 — Data Augmentation
Our combined dataset has around 12,000 audio files.
To make the model more robust and reduce overfitting, we create extra
training samples by slightly modifying existing ones:

- **Gaussian Noise** — adds very small random noise to the audio
- **Pitch Shifting** — shifts the pitch up or down slightly (±2 semitones)
- **Time Stretching** — speeds up or slows down the audio slightly (rate 0.8–1.2)

This effectively doubles or triples our training data without collecting new recordings.

### 3.4 — Label Encoding
Emotion labels are text (e.g. "happy", "sad").
We convert them to numbers (e.g. 0, 1, 2...) so the model can process them.

After this section, we have a clean, augmented, labelled set of audio files
ready for feature extraction in Section 4.

In [ ]:
# # ─────────────────────────────────────────────
# # SECTION 3 — PREPROCESSING + FEATURE EXTRACTION
# # (Memory Safe Version — No raw audio stored)
# # ─────────────────────────────────────────────

# import librosa
# import numpy as np
# from tqdm import tqdm

# SR = 22050
# DURATION = 3.0
# TARGET_LENGTH = int(SR * DURATION)
# N_MFCC = 40

# def extract_features(audio, sr):
#     """Extract all features from one audio array"""

#     # MFCC — 40 coefficients
#     mfcc = librosa.feature.mfcc(y=audio, sr=sr, n_mfcc=N_MFCC)
#     mfcc_mean = np.mean(mfcc, axis=1)
#     mfcc_std  = np.std(mfcc,  axis=1)

#     # Mel-spectrogram
#     mel = librosa.feature.melspectrogram(y=audio, sr=sr)
#     mel_mean = np.mean(mel, axis=1)

#     # Chroma
#     chroma = librosa.feature.chroma_stft(y=audio, sr=sr)
#     chroma_mean = np.mean(chroma, axis=1)

#     # Zero Crossing Rate
#     zcr = librosa.feature.zero_crossing_rate(audio)
#     zcr_mean = np.mean(zcr)

#     # Spectral Contrast
#     contrast = librosa.feature.spectral_contrast(y=audio, sr=sr)
#     contrast_mean = np.mean(contrast, axis=1)

#     # Tonnetz
#     harmonic = librosa.effects.harmonic(audio)
#     tonnetz = librosa.feature.tonnetz(y=harmonic, sr=sr)
#     tonnetz_mean = np.mean(tonnetz, axis=1)

#     # Combine all into one feature vector
#     features = np.concatenate([
#         mfcc_mean, mfcc_std,
#         mel_mean,
#         chroma_mean,
#         [zcr_mean],
#         contrast_mean,
#         tonnetz_mean
#     ])

#     return features


# def load_clean_extract(filepath, sr=SR, duration=DURATION):
#     """Load one file, clean it, extract features"""
#     try:
#         audio, sr_ = librosa.load(filepath, sr=sr, duration=duration)

#         # Trim silence
#         audio, _ = librosa.effects.trim(audio, top_db=20)

#         # Fix length
#         if len(audio) < TARGET_LENGTH:
#             audio = np.pad(audio, (0, TARGET_LENGTH - len(audio)))
#         else:
#             audio = audio[:TARGET_LENGTH]

#         # Normalise
#         if np.max(np.abs(audio)) > 0:
#             audio = audio / np.max(np.abs(audio))

#         return audio, sr_

#     except Exception:
#         return None, None


# def augment_and_extract(filepath, label):
#     """
#     Load one file, augment it 3 ways,
#     extract features from original + 3 augmented versions.
#     Returns list of (feature_vector, label)
#     """
#     results = []

#     audio, sr = load_clean_extract(filepath)
#     if audio is None:
#         return results

#     versions = []

#     # Original
#     versions.append(audio)

#     # Augmentation 1 — Gaussian noise
#     noisy = audio + 0.005 * np.random.randn(len(audio))
#     versions.append(noisy)

#     # Augmentation 2 — Pitch shift
#     try:
#         pitched = librosa.effects.pitch_shift(audio, sr=sr, n_steps=2)
#         pitched = pitched[:TARGET_LENGTH] if len(pitched) > TARGET_LENGTH else np.pad(pitched, (0, TARGET_LENGTH - len(pitched)))
#         versions.append(pitched)
#     except Exception:
#         versions.append(audio)

#     # Augmentation 3 — Time stretch
#     try:
#         stretched = librosa.effects.time_stretch(audio, rate=0.9)
#         stretched = stretched[:TARGET_LENGTH] if len(stretched) > TARGET_LENGTH else np.pad(stretched, (0, TARGET_LENGTH - len(stretched)))
#         versions.append(stretched)
#     except Exception:
#         versions.append(audio)

#     # Extract features from each version
#     for v in versions:
#         feat = extract_features(v, sr)
#         results.append((feat, label))

#     return results


# # ── Build full feature dataset ──

# print("Extracting features from all files (with augmentation)...")
# print("This will take 15-20 minutes. Please wait.\n")

# all_features = []
# all_labels   = []

# for _, row in tqdm(df.iterrows(), total=len(df)):
#     results = augment_and_extract(row['path'], row['emotion'])
#     for feat, label in results:
#         all_features.append(feat)
#         all_labels.append(label)

# # Convert to numpy arrays
# X = np.array(all_features)
# y = np.array(all_labels)

# print(f"\nFeature matrix shape:  {X.shape}")
# print(f"Labels shape:          {y.shape}")
# print(f"Unique emotions:       {np.unique(y)}")

In [ ]:
# ── Load saved features ──
import numpy as np
from sklearn.preprocessing import LabelEncoder, StandardScaler

X_scaled  = np.load('/kaggle/input/datasets/zavidd/ser-data/X_scaled.npy')
y_encoded = np.load('/kaggle/input/datasets/zavidd/ser-data/y_encoded.npy')
y         = np.load('/kaggle/input/datasets/zavidd/ser-data/y_labels.npy', allow_pickle=True)

# Rebuild label encoder
le = LabelEncoder()
le.fit(y)

print("Data loaded successfully.")
print(f"X_scaled shape : {X_scaled.shape}")
print(f"y_encoded shape: {y_encoded.shape}")
print(f"Emotions       : {le.classes_}")

In [ ]:
# ── Label Encoding ──
from sklearn.preprocessing import LabelEncoder, StandardScaler

le = LabelEncoder()
y_encoded = le.fit_transform(y)

print("Label encoding:")
for i, emotion in enumerate(le.classes_):
    print(f"  {emotion:12s} → {i}")

In [ ]:
# ── Save Section 3 Outputs ──

import numpy as np
import json
import pandas as pd
import matplotlib.pyplot as plt

# Save feature matrix and labels (already saved but save again to output folder)
np.save('/kaggle/working/outputs/section3_features/X_scaled.npy', X_scaled)
np.save('/kaggle/working/outputs/section3_features/y_encoded.npy', y_encoded)
np.save('/kaggle/working/outputs/section3_features/y_labels.npy', y)

# Save label encoding map as CSV
label_map = pd.DataFrame({
    'emotion' : le.classes_,
    'encoded' : list(range(len(le.classes_)))
})
label_map.to_csv(
    '/kaggle/working/outputs/section3_features/label_encoding.csv',
    index=False
)

# Save feature summary as JSON
summary_s3 = {
    'total_samples'      : int(X_scaled.shape[0]),
    'features_per_sample': int(X_scaled.shape[1]),
    'original_files'     : int(len(df)),
    'augmentation_factor': 4,
    'emotions'           : list(le.classes_),
    'feature_components' : {
        'mfcc_mean'       : 40,
        'mfcc_std'        : 40,
        'mel_spectrogram' : 128,
        'chroma'          : 12,
        'zcr'             : 1,
        'spectral_contrast': 7,
        'tonnetz'         : 6
    },
    'total_features': 234
}
with open('/kaggle/working/outputs/section3_features/feature_summary.json', 'w') as f:
    json.dump(summary_s3, f, indent=4)

# Plot emotion distribution after augmentation
import seaborn as sns
unique, counts = np.unique(y, return_counts=True)
plt.figure(figsize=(10, 4))
plt.bar(unique, counts, color='#2e7d32')
plt.title('Emotion Distribution After Augmentation')
plt.xlabel('Emotion')
plt.ylabel('Count')
plt.xticks(rotation=45)
plt.tight_layout()
plt.savefig(
    '/kaggle/working/outputs/section3_features/augmented_distribution.png',
    dpi=150
)
plt.close()

print("Section 3 outputs saved.")
print("  X_scaled.npy")
print("  y_encoded.npy")
print("  y_labels.npy")
print("  label_encoding.csv")
print("  feature_summary.json")
print("  augmented_distribution.png")

# week 3

## Section 4 — Baseline Model: SVM
---
Before building deep learning models, we first train a simple baseline model.

The baseline model is a Support Vector Machine (SVM) with an RBF kernel.
SVM is a classical machine learning classifier that works well with
numerical feature vectors like the ones we extracted.

Why do we need a baseline?
- It gives us a starting accuracy to compare against
- If our deep learning models do not beat this, something is wrong
- It proves that even simple models can work on this problem

We use an 80/10/10 train-validation-test split:
- 80% of data for training the model
- 10% for validation during development
- 10% for final testing and reporting results

Evaluation metrics used:
- Accuracy — overall correct predictions
- Weighted F1-score — handles class imbalance fairly
- Confusion Matrix — shows which emotions get confused with each other

In [ ]:
# ─────────────────────────────────────────────
# SECTION 4 — TRAIN TEST SPLIT + SVM BASELINE
# ─────────────────────────────────────────────

from sklearn.svm import SVC
from sklearn.model_selection import train_test_split
from sklearn.metrics import (accuracy_score, f1_score,
                             classification_report,
                             confusion_matrix)
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

# ── 4.1 Train / Validation / Test Split ──

# First split — 80% train, 20% temp
X_train, X_temp, y_train, y_temp = train_test_split(
    X_scaled, y_encoded,
    test_size=0.20,
    random_state=42,
    stratify=y_encoded
)

# Second split — split 20% into 10% val and 10% test
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp,
    test_size=0.50,
    random_state=42,
    stratify=y_temp
)

print("Dataset split complete:")
print(f"  Training samples   : {len(X_train)}")
print(f"  Validation samples : {len(X_val)}")
print(f"  Test samples       : {len(X_test)}")
print(f"  Total              : {len(X_train)+len(X_val)+len(X_test)}")

In [ ]:
# ── 4.2 Train SVM Baseline ──

print("Training SVM with RBF kernel...")
print("This will take 3-6 minutes. Please wait.\n")

svm_model = SVC(
    kernel='rbf',
    C=1.0,
    gamma='scale',
    random_state=42
)

svm_model.fit(X_train, y_train)
print("SVM training complete.")

In [ ]:
# ── 4.3 Evaluate SVM ──

y_pred_svm = svm_model.predict(X_test)

svm_accuracy = accuracy_score(y_test, y_pred_svm)
svm_f1       = f1_score(y_test, y_pred_svm, average='weighted')

print("=" * 45)
print("         SVM BASELINE RESULTS")
print("=" * 45)
print(f"  Accuracy        : {svm_accuracy * 100:.2f}%")
print(f"  Weighted F1     : {svm_f1:.4f}")
print("=" * 45)

print("\nPer-Class Report:")
print(classification_report(
    y_test, y_pred_svm,
    target_names=le.classes_
))

In [ ]:
# ── 4.4 Confusion Matrix ──

cm = confusion_matrix(y_test, y_pred_svm)

plt.figure(figsize=(10, 8))
sns.heatmap(
    cm,
    annot=True,
    fmt='d',
    cmap='Greens',
    xticklabels=le.classes_,
    yticklabels=le.classes_
)
plt.title('SVM Baseline — Confusion Matrix',
          fontsize=14, fontweight='bold')
plt.xlabel('Predicted Emotion')
plt.ylabel('True Emotion')
plt.tight_layout()
plt.show()

In [ ]:
# ── 4.5 Save All Section 4 Outputs ──

import os
import json
import joblib
import pandas as pd

# Create Section 4 output folder
os.makedirs('/kaggle/working/outputs/section4_svm', exist_ok=True)

# ── Save model ──
joblib.dump(svm_model, '/kaggle/working/outputs/section4_svm/svm_model.pkl')

# ── Save confusion matrix plot ──
fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(
    cm, annot=True, fmt='d', cmap='Greens',
    xticklabels=le.classes_,
    yticklabels=le.classes_,
    ax=ax
)
ax.set_title('SVM Baseline — Confusion Matrix', fontsize=14, fontweight='bold')
ax.set_xlabel('Predicted Emotion')
ax.set_ylabel('True Emotion')
plt.tight_layout()
plt.savefig('/kaggle/working/outputs/section4_svm/svm_confusion_matrix.png', dpi=150)
plt.close()

# ── Save results as CSV ──
report_dict = classification_report(
    y_test, y_pred_svm,
    target_names=le.classes_,
    output_dict=True
)
report_df = pd.DataFrame(report_dict).transpose()
report_df.to_csv('/kaggle/working/outputs/section4_svm/svm_classification_report.csv')

# ── Save summary as JSON ──
summary = {
    'model'       : 'SVM RBF Kernel',
    'section'     : 'Section 4 — Baseline',
    'accuracy'    : round(svm_accuracy * 100, 2),
    'weighted_f1' : round(svm_f1, 4),
    'train_size'  : len(X_train),
    'val_size'    : len(X_val),
    'test_size'   : len(X_test),
    'per_class'   : {
        cls: {
            'precision' : round(report_dict[cls]['precision'], 4),
            'recall'    : round(report_dict[cls]['recall'], 4),
            'f1-score'  : round(report_dict[cls]['f1-score'], 4)
        }
        for cls in le.classes_
    }
}

with open('/kaggle/working/outputs/section4_svm/svm_results.json', 'w') as f:
    json.dump(summary, f, indent=4)

# ── Print confirmation ──
print("Section 4 outputs saved successfully.")
print("\nFiles saved to /kaggle/working/outputs/section4_svm/")
print("  svm_model.pkl                — trained SVM model")
print("  svm_confusion_matrix.png     — confusion matrix plot")
print("  svm_classification_report.csv — per class metrics")
print("  svm_results.json             — summary results")
print(f"\n  Accuracy    : {svm_accuracy * 100:.2f}%")
print(f"  Weighted F1 : {svm_f1:.4f}")

# week 4

## Section 5 — Deep Learning Model 1: 1D-CNN
---
In this section we build our first deep learning model — a 1D Convolutional
Neural Network (1D-CNN).

What is a CNN?
A CNN learns local patterns from data. In our case it learns which parts
of the feature vector are most important for detecting emotions.
It is much more powerful than SVM because it learns features automatically
rather than relying on fixed rules.

Why 1D-CNN?
Our features are a 1D vector (234 numbers per sample).
A 1D-CNN slides small filters across this vector to detect patterns,
similar to how it detects edges in images but for audio features instead.

Expected accuracy: 85-90%
This should be significantly better than our SVM baseline of 75.12%

In [ ]:
# ─────────────────────────────────────────────
# SECTION 5 — 1D-CNN MODEL
# ─────────────────────────────────────────────

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import (accuracy_score, f1_score,
                             classification_report,
                             confusion_matrix)

# ── 5.1 Device Setup ──
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")

# ── 5.2 Dataset Class ──
class EmotionDataset(Dataset):
    def __init__(self, X, y):
        # CNN expects shape (batch, channels, length)
        self.X = torch.FloatTensor(X).unsqueeze(1)
        self.y = torch.LongTensor(y)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

# ── 5.3 Create DataLoaders ──
BATCH_SIZE = 64

train_dataset = EmotionDataset(X_train, y_train)
val_dataset   = EmotionDataset(X_val,   y_val)
test_dataset  = EmotionDataset(X_test,  y_test)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader   = DataLoader(val_dataset,   batch_size=BATCH_SIZE, shuffle=False)
test_loader  = DataLoader(test_dataset,  batch_size=BATCH_SIZE, shuffle=False)

print(f"Train batches : {len(train_loader)}")
print(f"Val batches   : {len(val_loader)}")
print(f"Test batches  : {len(test_loader)}")

In [ ]:
# ── 5.4 Define 1D-CNN Architecture ──

class CNN1D(nn.Module):
    def __init__(self, input_size, num_classes):
        super(CNN1D, self).__init__()

        # Convolutional Block 1
        self.conv1 = nn.Sequential(
            nn.Conv1d(1, 64, kernel_size=3, padding=1),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.MaxPool1d(kernel_size=2),
            nn.Dropout(0.25)
        )

        # Convolutional Block 2
        self.conv2 = nn.Sequential(
            nn.Conv1d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.MaxPool1d(kernel_size=2),
            nn.Dropout(0.25)
        )

        # Convolutional Block 3
        self.conv3 = nn.Sequential(
            nn.Conv1d(128, 256, kernel_size=3, padding=1),
            nn.BatchNorm1d(256),
            nn.ReLU(),
            nn.MaxPool1d(kernel_size=2),
            nn.Dropout(0.25)
        )

        # Calculate flattened size
        self.flat_size = 256 * (input_size // 8)

        # Fully Connected Layers
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(self.flat_size, 256),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(128, num_classes)
        )

    def forward(self, x):
        x = self.conv1(x)
        x = self.conv2(x)
        x = self.conv3(x)
        x = self.classifier(x)
        return x


# Build model
NUM_CLASSES  = len(le.classes_)
INPUT_SIZE   = X_train.shape[1]

cnn_model = CNN1D(INPUT_SIZE, NUM_CLASSES).to(device)

print("CNN Model Architecture:")
print(cnn_model)
print(f"\nInput size  : {INPUT_SIZE}")
print(f"Num classes : {NUM_CLASSES}")

# Count parameters
total_params = sum(p.numel() for p in cnn_model.parameters())
print(f"Total parameters: {total_params:,}")

In [ ]:
# ── 5.5 Train CNN ──

EPOCHS    = 30
LR        = 0.001

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(cnn_model.parameters(), lr=LR)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='min', factor=0.5, patience=3
)

# Track history
cnn_history = {
    'train_loss': [], 'val_loss': [],
    'train_acc' : [], 'val_acc' : []
}

best_val_loss = float('inf')
best_model_state = None

print(f"Training 1D-CNN for {EPOCHS} epochs...")
print("-" * 55)

for epoch in range(EPOCHS):

    # ── Training ──
    cnn_model.train()
    train_loss, train_correct, train_total = 0, 0, 0

    for X_batch, y_batch in train_loader:
        X_batch = X_batch.to(device)
        y_batch = y_batch.to(device)

        optimizer.zero_grad()
        outputs = cnn_model(X_batch)
        loss    = criterion(outputs, y_batch)
        loss.backward()
        optimizer.step()

        train_loss    += loss.item()
        preds          = outputs.argmax(dim=1)
        train_correct += (preds == y_batch).sum().item()
        train_total   += len(y_batch)

    # ── Validation ──
    cnn_model.eval()
    val_loss, val_correct, val_total = 0, 0, 0

    with torch.no_grad():
        for X_batch, y_batch in val_loader:
            X_batch = X_batch.to(device)
            y_batch = y_batch.to(device)

            outputs    = cnn_model(X_batch)
            loss       = criterion(outputs, y_batch)
            val_loss  += loss.item()
            preds      = outputs.argmax(dim=1)
            val_correct += (preds == y_batch).sum().item()
            val_total   += len(y_batch)

    # ── Metrics ──
    avg_train_loss = train_loss / len(train_loader)
    avg_val_loss   = val_loss   / len(val_loader)
    train_acc      = train_correct / train_total * 100
    val_acc        = val_correct   / val_total   * 100

    cnn_history['train_loss'].append(avg_train_loss)
    cnn_history['val_loss'].append(avg_val_loss)
    cnn_history['train_acc'].append(train_acc)
    cnn_history['val_acc'].append(val_acc)

    # Save best model
    if avg_val_loss < best_val_loss:
        best_val_loss    = avg_val_loss
        best_model_state = cnn_model.state_dict().copy()

    scheduler.step(avg_val_loss)

    print(f"Epoch [{epoch+1:02d}/{EPOCHS}] "
          f"Train Loss: {avg_train_loss:.4f} "
          f"Train Acc: {train_acc:.2f}% "
          f"Val Loss: {avg_val_loss:.4f} "
          f"Val Acc: {val_acc:.2f}%")

print("-" * 55)
print("Training complete.")

# Load best model
cnn_model.load_state_dict(best_model_state)
print("Best model loaded.")

In [ ]:
# ── 5.6 Evaluate CNN on Test Set ──

cnn_model.eval()
all_preds, all_labels = [], []

with torch.no_grad():
    for X_batch, y_batch in test_loader:
        X_batch  = X_batch.to(device)
        outputs  = cnn_model(X_batch)
        preds    = outputs.argmax(dim=1).cpu().numpy()
        all_preds.extend(preds)
        all_labels.extend(y_batch.numpy())

cnn_accuracy = accuracy_score(all_labels, all_preds)
cnn_f1       = f1_score(all_labels, all_preds, average='weighted')

print("=" * 45)
print("         1D-CNN RESULTS")
print("=" * 45)
print(f"  Accuracy        : {cnn_accuracy * 100:.2f}%")
print(f"  Weighted F1     : {cnn_f1:.4f}")
print("=" * 45)

print("\nPer-Class Report:")
print(classification_report(
    all_labels, all_preds,
    target_names=le.classes_
))

In [ ]:
# ── 5.7 Plot Training History ──

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Loss plot
axes[0].plot(cnn_history['train_loss'], label='Train Loss', color='#2e7d32')
axes[0].plot(cnn_history['val_loss'],   label='Val Loss',   color='#ff6f00')
axes[0].set_title('CNN — Loss Curve')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].legend()

# Accuracy plot
axes[1].plot(cnn_history['train_acc'], label='Train Acc', color='#2e7d32')
axes[1].plot(cnn_history['val_acc'],   label='Val Acc',   color='#ff6f00')
axes[1].set_title('CNN — Accuracy Curve')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Accuracy (%)')
axes[1].legend()

plt.tight_layout()
plt.savefig('/kaggle/working/outputs/section5_cnn/cnn_training_curves.png', dpi=150)
plt.show()

# ── Confusion Matrix ──
cm_cnn = confusion_matrix(all_labels, all_preds)

plt.figure(figsize=(10, 8))
sns.heatmap(
    cm_cnn, annot=True, fmt='d', cmap='Greens',
    xticklabels=le.classes_,
    yticklabels=le.classes_
)
plt.title('1D-CNN — Confusion Matrix', fontsize=14, fontweight='bold')
plt.xlabel('Predicted Emotion')
plt.ylabel('True Emotion')
plt.tight_layout()
plt.savefig('/kaggle/working/outputs/section5_cnn/cnn_confusion_matrix.png', dpi=150)
plt.show()

# ── Save Model ──
import json, pandas as pd
torch.save(best_model_state,
           '/kaggle/working/outputs/section5_cnn/cnn_model.pth')

# ── Save CSV Report ──
report_dict = classification_report(
    all_labels, all_preds,
    target_names=le.classes_,
    output_dict=True
)
pd.DataFrame(report_dict).transpose().to_csv(
    '/kaggle/working/outputs/section5_cnn/cnn_classification_report.csv'
)

# ── Save JSON Summary ──
summary_cnn = {
    'model'      : '1D-CNN',
    'section'    : 'Section 5',
    'accuracy'   : round(cnn_accuracy * 100, 2),
    'weighted_f1': round(cnn_f1, 4),
    'epochs'     : EPOCHS,
    'batch_size' : BATCH_SIZE,
    'optimizer'  : 'Adam',
    'lr'         : LR,
    'per_class'  : {
        cls: {
            'precision': round(report_dict[cls]['precision'], 4),
            'recall'   : round(report_dict[cls]['recall'],    4),
            'f1-score' : round(report_dict[cls]['f1-score'],  4)
        }
        for cls in le.classes_
    }
}
with open('/kaggle/working/outputs/section5_cnn/cnn_results.json', 'w') as f:
    json.dump(summary_cnn, f, indent=4)

print("Section 5 outputs saved.")
print("  ✓  cnn_model.pth")
print("  ✓  cnn_training_curves.png")
print("  ✓  cnn_confusion_matrix.png")
print("  ✓  cnn_classification_report.csv")
print("  ✓  cnn_results.json")
print(f"\n  Accuracy    : {cnn_accuracy * 100:.2f}%")
print(f"  Weighted F1 : {cnn_f1:.4f}")

## Section 6 — Deep Learning Model 2: CNN-LSTM
---
In this section we build a CNN-LSTM hybrid model.

What is CNN-LSTM?
- CNN part learns local patterns from the feature vector
- LSTM part learns how those patterns change over time
- Together they capture both the shape and the sequence of emotions in speech

Why is CNN-LSTM better than CNN alone?
- Emotions in speech are not just about what frequencies exist
- They also depend on HOW those frequencies change over time
- LSTM is specifically designed to learn these time-based patterns
- Combining both gives a much richer understanding of the speech signal

Expected accuracy: 88-92%
This should be better than our 1D-CNN result of 81.85%

In [ ]:
# ─────────────────────────────────────────────
# SECTION 6 — CNN-LSTM MODEL
# ─────────────────────────────────────────────

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import (accuracy_score, f1_score,
                             classification_report,
                             confusion_matrix)
import json
import pandas as pd

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")

# ── 6.1 Define CNN-LSTM Architecture ──

class CNNLSTM(nn.Module):
    def __init__(self, input_size, num_classes,
                 lstm_hidden=128, lstm_layers=2):
        super(CNNLSTM, self).__init__()

        # CNN Feature Extractor
        self.cnn = nn.Sequential(
            # Block 1
            nn.Conv1d(1, 64, kernel_size=3, padding=1),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.MaxPool1d(kernel_size=2),
            nn.Dropout(0.25),

            # Block 2
            nn.Conv1d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.MaxPool1d(kernel_size=2),
            nn.Dropout(0.25),

            # Block 3
            nn.Conv1d(128, 256, kernel_size=3, padding=1),
            nn.BatchNorm1d(256),
            nn.ReLU(),
            nn.MaxPool1d(kernel_size=2),
            nn.Dropout(0.25)
        )

        # Calculate CNN output size
        self.cnn_out_size = input_size // 8

        # LSTM Temporal Modelling
        self.lstm = nn.LSTM(
            input_size  = 256,
            hidden_size = lstm_hidden,
            num_layers  = lstm_layers,
            batch_first = True,
            dropout     = 0.3,
            bidirectional = False
        )

        # Classifier
        self.classifier = nn.Sequential(
            nn.Linear(lstm_hidden, 128),
            nn.ReLU(),
            nn.Dropout(0.4),
            nn.Linear(128, num_classes)
        )

    def forward(self, x):
        # x shape: (batch, 1, features)

        # CNN extracts local patterns
        x = self.cnn(x)

        # Reshape for LSTM: (batch, seq_len, channels)
        x = x.permute(0, 2, 1)

        # LSTM learns temporal patterns
        lstm_out, _ = self.lstm(x)

        # Take last timestep output
        x = lstm_out[:, -1, :]

        # Classify
        x = self.classifier(x)
        return x


# Build model
NUM_CLASSES = len(le.classes_)
INPUT_SIZE  = X_train.shape[1]

cnnlstm_model = CNNLSTM(INPUT_SIZE, NUM_CLASSES).to(device)

print("CNN-LSTM Model Architecture:")
print(cnnlstm_model)

total_params = sum(p.numel() for p in cnnlstm_model.parameters())
print(f"\nTotal parameters: {total_params:,}")

In [ ]:
# ── 6.2 Train CNN-LSTM ──

EPOCHS    = 30
LR        = 0.001
BATCH_SIZE = 64

# DataLoaders (reuse same splits)
train_loader = DataLoader(train_dataset,
                          batch_size=BATCH_SIZE, shuffle=True)
val_loader   = DataLoader(val_dataset,
                          batch_size=BATCH_SIZE, shuffle=False)
test_loader  = DataLoader(test_dataset,
                          batch_size=BATCH_SIZE, shuffle=False)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(cnnlstm_model.parameters(), lr=LR)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='min', factor=0.5, patience=3
)

cnnlstm_history = {
    'train_loss': [], 'val_loss': [],
    'train_acc' : [], 'val_acc' : []
}

best_val_loss     = float('inf')
best_model_state  = None

print(f"Training CNN-LSTM for {EPOCHS} epochs...")
print("-" * 60)

for epoch in range(EPOCHS):

    # ── Training ──
    cnnlstm_model.train()
    train_loss, train_correct, train_total = 0, 0, 0

    for X_batch, y_batch in train_loader:
        X_batch = X_batch.to(device)
        y_batch = y_batch.to(device)

        optimizer.zero_grad()
        outputs = cnnlstm_model(X_batch)
        loss    = criterion(outputs, y_batch)
        loss.backward()
        nn.utils.clip_grad_norm_(cnnlstm_model.parameters(), 1.0)
        optimizer.step()

        train_loss    += loss.item()
        preds          = outputs.argmax(dim=1)
        train_correct += (preds == y_batch).sum().item()
        train_total   += len(y_batch)

    # ── Validation ──
    cnnlstm_model.eval()
    val_loss, val_correct, val_total = 0, 0, 0

    with torch.no_grad():
        for X_batch, y_batch in val_loader:
            X_batch = X_batch.to(device)
            y_batch = y_batch.to(device)

            outputs     = cnnlstm_model(X_batch)
            loss        = criterion(outputs, y_batch)
            val_loss   += loss.item()
            preds       = outputs.argmax(dim=1)
            val_correct += (preds == y_batch).sum().item()
            val_total   += len(y_batch)

    # ── Metrics ──
    avg_train_loss = train_loss / len(train_loader)
    avg_val_loss   = val_loss   / len(val_loader)
    train_acc      = train_correct / train_total * 100
    val_acc        = val_correct   / val_total   * 100

    cnnlstm_history['train_loss'].append(avg_train_loss)
    cnnlstm_history['val_loss'].append(avg_val_loss)
    cnnlstm_history['train_acc'].append(train_acc)
    cnnlstm_history['val_acc'].append(val_acc)

    if avg_val_loss < best_val_loss:
        best_val_loss    = avg_val_loss
        best_model_state = cnnlstm_model.state_dict().copy()

    scheduler.step(avg_val_loss)

    print(f"Epoch [{epoch+1:02d}/{EPOCHS}] "
          f"Train Loss: {avg_train_loss:.4f} "
          f"Train Acc: {train_acc:.2f}% "
          f"Val Loss: {avg_val_loss:.4f} "
          f"Val Acc: {val_acc:.2f}%")

print("-" * 60)
print("Training complete.")
cnnlstm_model.load_state_dict(best_model_state)
print("Best model loaded.")

In [ ]:
# ── 6.3 Evaluate CNN-LSTM ──

cnnlstm_model.eval()
all_preds, all_labels = [], []

with torch.no_grad():
    for X_batch, y_batch in test_loader:
        X_batch = X_batch.to(device)
        outputs = cnnlstm_model(X_batch)
        preds   = outputs.argmax(dim=1).cpu().numpy()
        all_preds.extend(preds)
        all_labels.extend(y_batch.numpy())

cnnlstm_accuracy = accuracy_score(all_labels, all_preds)
cnnlstm_f1       = f1_score(all_labels, all_preds, average='weighted')

print("=" * 45)
print("       CNN-LSTM RESULTS")
print("=" * 45)
print(f"  Accuracy        : {cnnlstm_accuracy * 100:.2f}%")
print(f"  Weighted F1     : {cnnlstm_f1:.4f}")
print("=" * 45)

print("\nPer-Class Report:")
print(classification_report(
    all_labels, all_preds,
    target_names=le.classes_
))

In [ ]:
# ── 6.4 Plot and Save Section 6 Outputs ──

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(cnnlstm_history['train_loss'],
             label='Train Loss', color='#2e7d32')
axes[0].plot(cnnlstm_history['val_loss'],
             label='Val Loss',   color='#ff6f00')
axes[0].set_title('CNN-LSTM — Loss Curve')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].legend()

axes[1].plot(cnnlstm_history['train_acc'],
             label='Train Acc', color='#2e7d32')
axes[1].plot(cnnlstm_history['val_acc'],
             label='Val Acc',   color='#ff6f00')
axes[1].set_title('CNN-LSTM — Accuracy Curve')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Accuracy (%)')
axes[1].legend()

plt.tight_layout()
plt.savefig(
    '/kaggle/working/outputs/section6_cnn_lstm/cnnlstm_training_curves.png',
    dpi=150
)
plt.show()

# Confusion Matrix
cm_cnnlstm = confusion_matrix(all_labels, all_preds)
plt.figure(figsize=(10, 8))
sns.heatmap(
    cm_cnnlstm, annot=True, fmt='d', cmap='Greens',
    xticklabels=le.classes_,
    yticklabels=le.classes_
)
plt.title('CNN-LSTM — Confusion Matrix', fontsize=14, fontweight='bold')
plt.xlabel('Predicted Emotion')
plt.ylabel('True Emotion')
plt.tight_layout()
plt.savefig(
    '/kaggle/working/outputs/section6_cnn_lstm/cnnlstm_confusion_matrix.png',
    dpi=150
)
plt.show()

# Save model
torch.save(
    best_model_state,
    '/kaggle/working/outputs/section6_cnn_lstm/cnnlstm_model.pth'
)

# Save CSV report
report_dict = classification_report(
    all_labels, all_preds,
    target_names=le.classes_,
    output_dict=True
)
pd.DataFrame(report_dict).transpose().to_csv(
    '/kaggle/working/outputs/section6_cnn_lstm/cnnlstm_classification_report.csv'
)

# Save JSON summary
summary_cnnlstm = {
    'model'      : 'CNN-LSTM',
    'section'    : 'Section 6',
    'accuracy'   : round(cnnlstm_accuracy * 100, 2),
    'weighted_f1': round(cnnlstm_f1, 4),
    'epochs'     : EPOCHS,
    'batch_size' : BATCH_SIZE,
    'optimizer'  : 'Adam',
    'lr'         : LR,
    'per_class'  : {
        cls: {
            'precision': round(report_dict[cls]['precision'], 4),
            'recall'   : round(report_dict[cls]['recall'],    4),
            'f1-score' : round(report_dict[cls]['f1-score'],  4)
        }
        for cls in le.classes_
    }
}
with open(
    '/kaggle/working/outputs/section6_cnn_lstm/cnnlstm_results.json', 'w'
) as f:
    json.dump(summary_cnnlstm, f, indent=4)

print("Section 6 outputs saved.")
print("  ✓  cnnlstm_model.pth")
print("  ✓  cnnlstm_training_curves.png")
print("  ✓  cnnlstm_confusion_matrix.png")
print("  ✓  cnnlstm_classification_report.csv")
print("  ✓  cnnlstm_results.json")
print(f"\n  Accuracy    : {cnnlstm_accuracy * 100:.2f}%")
print(f"  Weighted F1 : {cnnlstm_f1:.4f}")